# ATC Multi-Agent GRPO Training

**Runtime**: `Runtime → Change runtime type → GPU (T4 or better)`

**Default model**: `Qwen2.5-1.5B-Instruct` — safe on T4 (16 GB).  
Change `MODEL_NAME` to `Qwen/Qwen2.5-7B-Instruct` for L4 / A100.

**Works on**: Google Colab · Kaggle Notebooks

**Parity:** Training defaults, env vars, and `train_grpo.py` / `train_sft.py` flags match `training/train_jupyter.ipynb` §0 + §9 (Colab-specific paths and streaming logs differ).

### What this notebook does
1. Installs dependencies in the correct order (fixes the `retry` import error)
2. Clones the ATC repo
3. Verifies reward diversity (constant-reward bugs are fixed in source)
4. Runs SFT (optional) then GRPO with the same flags as `train_jupyter` (default `adapt_multidomain` = domain-only AMAN+DMAN+ADAPT; grounded curriculum off)
5. Plots reward curves (labelled axes, EMA smoothing, std collapse detector)
6. Compares trained vs heuristic baseline side-by-side
7. Saves all plots as `.png` for README embedding

## 1. Configure

In [ ]:
import os, sys
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle/working')
PLATFORM  = 'colab' if IS_COLAB else ('kaggle' if IS_KAGGLE else 'local')
print(f'Platform: {PLATFORM}')

# ── Repository ────────────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/GTsingh600/ats.git'
REPO_BRANCH = 'main'

# ── Model (match training/train_jupyter.ipynb: use 7B when VRAM allows) ───────
MODEL_NAME    = 'Qwen/Qwen2.5-7B-Instruct'  # Colab T4-safe; set to Qwen/Qwen2.5-7B-Instruct for A100-class

# ── Training (defaults aligned with training/train_jupyter.ipynb §0) ─────────
EPISODES      = 100
N_GENERATIONS = 4
LORA_RANK     = 16
SEED          = 42
BATCH_SIZE    = 8
GRAD_ACCUM    = 2
MAX_NEW_TOKENS  = 384
TEMPERATURE     = 0.7
LOGGING_STEPS   = 1
EVAL_EPISODES   = 3

# GRPO roster / dataset (same CLI as train_jupyter)
TRAINING_MODE = 'adapt_multidomain'  # 'full' | 'hyper_minimal' | 'adapt_multidomain'
RELAX_ROSTER = True
USE_GROUNDED_CURRICULUM = False  # must stay False for adapt_multidomain
STATIC_GROUNDED_DATASET = False
ATC_LIVE_PASSES = 2.5
ATC_SAVE_STEPS = 20
SKIP_MAIN_EVAL = True
STRICT_GATES = True
CURRICULUM_STATE = None  # optional path str to grounded_curriculum_state.json

# SFT pre-GRPO (same as train_jupyter defaults)
RUN_SFT_PHASE = True
SFT_N_EPISODES = 200
SFT_MAX_STEPS = 300
SFT_BATCH = 2
SFT_GRAD_ACCUM = 4
SFT_EVAL_SPLIT = 0.0
SFT_EARLY_STOP_PATIENCE = 0

# ── W&B (optional) ────────────────────────────────────────────────────────────
WANDB_API_KEY = ''
WANDB_PROJECT = 'atc-multiagent-grpo'

# ── Paths ─────────────────────────────────────────────────────────────────────
if IS_COLAB:
    REPO_DIR   = Path('/content/ats')
    OUTPUT_DIR = Path('/content/drive/MyDrive/atc-outputs')
elif IS_KAGGLE:
    REPO_DIR   = Path('/kaggle/working/ats')
    OUTPUT_DIR = Path('/kaggle/working/atc-outputs')
else:
    REPO_DIR   = Path('./ats')
    OUTPUT_DIR = Path('./atc-outputs')

PLOTS_DIR = OUTPUT_DIR / 'plots'
SFT_OUTPUT_DIR = OUTPUT_DIR / 'atc-sft-json'

os.environ['ATC_TRAINING_MODE'] = str(TRAINING_MODE)
print(f'Model:      {MODEL_NAME}')
print(f'Episodes:   {EPISODES}  TRAINING_MODE={TRAINING_MODE}  RELAX_ROSTER={RELAX_ROSTER}')
print(f'Grounded:   live={USE_GROUNDED_CURRICULUM and not STATIC_GROUNDED_DATASET} static={STATIC_GROUNDED_DATASET}')
print(f'SFT:        RUN_SFT_PHASE={RUN_SFT_PHASE} max_steps={SFT_MAX_STEPS} n_episodes={SFT_N_EPISODES}')
print(f'Output:     {OUTPUT_DIR}')


## 2. Mount storage

In [ ]:
if IS_COLAB:
    USE_DRIVE = True
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        OUTPUT_DIR = Path('/content/atc-outputs')
        PLOTS_DIR  = OUTPUT_DIR / 'plots'
elif IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        if not WANDB_API_KEY:
            WANDB_API_KEY = _s.get_secret('WANDB_API_KEY')
        print('Loaded WANDB_API_KEY from Kaggle secrets')
    except Exception:
        pass

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output ready: {OUTPUT_DIR}')

## 3. Install dependencies

**Install order matters.**  
Unsloth is installed **first without `--no-deps`** so pip can resolve a compatible
`transformers` version. Installing with `--no-deps` causes the
`ImportError: cannot import name 'retry' from transformers.utils.generic` error
because the pre-installed Colab/Kaggle `transformers` may not match what
`unsloth_zoo` requires.  
After unsloth pins `transformers`, we reinstall `trl` and other packages.

In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

def _pip_uninstall(*pkgs):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', *pkgs],
        capture_output=True,
    )

# ── Step 0: Remove known conflicts ───────────────────────────────────────────
_pip_uninstall('vllm', 'vllm-flash-attn')
print('Step 0: removed conflicting packages')

# ── Step 1: Unsloth WITH deps ─────────────────────────────────────────────────
# CRITICAL: do NOT add --no-deps here.
# unsloth_zoo imports `retry` from transformers.utils.generic; the version of
# transformers that exposes this is pulled in by unsloth's own requirements.
# Skipping deps leaves whatever Colab pre-installed, which causes ImportError.
_pip('unsloth==2026.4.7', 'unsloth-zoo==2026.4.9')
print('Step 1: unsloth installed (with deps, including compatible transformers)')

# ── Step 2: TRL and training stack ───────────────────────────────────────────
# Install AFTER unsloth so transformers is already at the right version.
# --no-deps for these is safe because unsloth already resolved the graph.
_pip(
    'trl==0.16.0',
    'accelerate==1.13.0',
    'peft==0.19.1',
    'bitsandbytes==0.49.2',
    'datasets==2.20.0',
    '--no-deps',
)
print('Step 2: trl + training stack installed')

_pip('mergekit')
print('Step 2b: mergekit (TRL GRPO import chain)')



# ── Step 3: HuggingFace utilities ────────────────────────────────────────────
_pip(
    'huggingface-hub==0.36.2',
    'hf_transfer==0.1.9',
    'tyro==0.9.17',
    'multiprocess==0.70.17',
    '--no-deps',
)
print('Step 3: HuggingFace utilities installed')

# ── Step 4: W&B + OpenEnv + plotting ────────────────────────────────────────
_pip('wandb>=0.18,<0.20')
_pip(
    'openenv-core>=0.2.3',
    'fastapi>=0.128.0',
    'pydantic>=2.12.0',
    'uvicorn>=0.41.0',
    'matplotlib>=3.9.0',
    'numpy>=1.26.0',
)
print('Step 4: W&B + OpenEnv + plotting installed')

# ── Global env flags ─────────────────────────────────────────────────────────
import os
os.environ['TORCH_COMPILE_DISABLE']  = '1'   # avoid Dynamo issues with GRPO
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('\nInstallation complete.')

## 4. Clone repository

In [ ]:
import shutil, os, sys
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1',
     REPO_URL, str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'Working directory: {Path.cwd()}')

# Clear stale __pycache__ so Unsloth rebuilds compiled kernels fresh
for d in REPO_DIR.rglob('__pycache__'):
    shutil.rmtree(d, ignore_errors=True)
print('Cleared __pycache__')

## 5. Verify environment

In [ ]:
import torch

print(f'PyTorch:      {torch.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM:         {vram_gb:.1f} GB')
    if vram_gb < 14:
        print('WARNING: <14 GB VRAM detected. Use 1.5B model + N_GENERATIONS=2')

_unsloth_ver = None
try:
    import unsloth  # noqa: F401
    _unsloth_ver = unsloth.__version__
except Exception as exc:
    print(f'Unsloth preload: {exc}')

import transformers
import trl
import peft
print(f'Transformers: {transformers.__version__}')
if _unsloth_ver is not None:
    print(f'Unsloth:      {_unsloth_ver}')
print(f'TRL:          {trl.__version__}')
print(f'PEFT:         {peft.__version__}')

try:
    from transformers.utils.generic import retry  # noqa: F401
    print('transformers.utils.generic.retry: OK')
except ImportError:
    print('WARNING: retry not in transformers.utils.generic')
    print('  Re-run the install cell.')

from trl import GRPOConfig, GRPOTrainer
print('GRPOConfig + GRPOTrainer: OK')
if _unsloth_ver is not None:
    from unsloth import FastLanguageModel  # noqa: F401
    print('FastLanguageModel: OK')
else:
    print('FastLanguageModel: skipped (unsloth not loaded)')

from training.dataset import build_episode_dataset
_check = build_episode_dataset(n_episodes=2, seed=0, training_mode=TRAINING_MODE)
print(f'Dataset builder: OK — {len(_check)} samples, roles={sorted({r["agent_role"] for r in _check})}')


## 6. Verify reward diversity

Three bugs caused constant AMAN/DMAN rewards (zero GRPO gradient). All fixed in source:

| Bug | Before | After |
|-----|--------|-------|
| `delay_eff` for empty plan | `1.0` (wrong) | `0.0` |
| Parse failure reward | fixed `-0.8` → group std=0 | content-aware `[-0.80, -0.62]` |
| Generation temperature | `0.7` (low diversity) | `0.9` |

In [ ]:
import statistics
from training.reward_functions import aman_reward_fn, _parse_partial_credit
from tasks import task_catalog

catalog = task_catalog()
tid     = list(catalog.keys())[0]
print(f'Test task: {tid} ({len(catalog[tid].flights)} flights)')

# Test 1: empty arrival_slots must NOT get delay_eff=1.0
r_empty = aman_reward_fn(['{"arrival_slots": [], "rationale": "test"}'], task_id=[tid])
print(f'\nEmpty plan reward: {r_empty[0]:.4f}')
assert r_empty[0] < 0.20, f'Bug still present: empty plan scored {r_empty[0]:.3f} (expect < 0.20)'
print('  PASS: empty plan correctly penalised')

# Test 2: group of 4 completions must have std > 0
group = [
    'no json here at all',
    '{"arrival_slots": [{"flight_id": "AI101", "runway": "28L", "assigned_minute": 30, "hold_minutes": 0}], "rationale": "ok"}',
    '{"arrival_slots": [], "rationale": "runway unavailable"}',
    '{flight_id: broken json',
]
rewards = aman_reward_fn(group, task_id=[tid]*4)
std = statistics.stdev(rewards)
print(f'\nGroup rewards: {[round(r,4) for r in rewards]}')
print(f'Group std:     {std:.4f}')
assert std > 0, 'FAIL: all rewards identical — GRPO gets zero gradient'
print('  PASS: reward diversity confirmed (GRPO gradient will flow)')

# Test 3: _parse_partial_credit is monotonically more generous with more structure
samples = [
    ('empty',             ''),
    ('plain text',        'I cannot do this'),
    ('has brace',         '{some content}'),
    ('has arrival_slots', '{"arrival_slots": [...]}'),
    ('has flight+runway', '{"arrival_slots": [{"flight_id":"X","runway":"28L"}]}'),
]
print('\n_parse_partial_credit:')
prev = -1
for label, text in samples:
    c = _parse_partial_credit(text)
    print(f'  {label:25s}: {c:.4f}')
    assert c >= prev, f'Expected non-decreasing credit, got {c} < {prev}'
    prev = c

print('\nAll reward diversity checks PASSED.')

## 7. W&B setup (optional)

In [ ]:
import os

if IS_COLAB and not WANDB_API_KEY:
    try:
        from google.colab import userdata
        WANDB_API_KEY = userdata.get('WANDB_API_KEY')
        print('Loaded WANDB_API_KEY from Colab secrets')
    except Exception:
        pass

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_MODE']    = 'online'
    print(f'W&B: online  project={WANDB_PROJECT}')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B: disabled (set WANDB_API_KEY to enable)')

## 8. Train

`train_grpo.py` runs: load 4-bit QLoRA model → build episode dataset → GRPO loop → save checkpoint + curves.

> **OOM?** Lower `N_GENERATIONS` to `2`, or switch to `Qwen2.5-1.5B-Instruct`.

In [ ]:
import time, subprocess, sys, os, re
from collections import defaultdict
from pathlib import Path

SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _train_env():
    e = os.environ.copy()
    e['PYTHONUNBUFFERED'] = '1'
    e['ATC_STRICT_GATES'] = '1' if STRICT_GATES else '0'
    e['ATC_TRAINING_MODE'] = str(TRAINING_MODE)
    e['ATC_LIVE_PASSES'] = str(float(ATC_LIVE_PASSES))
    e['ATC_SAVE_STEPS'] = str(int(ATC_SAVE_STEPS))
    if STATIC_GROUNDED_DATASET:
        e['ATC_STATIC_GROUNDED_DATASET'] = '1'
    else:
        e.pop('ATC_STATIC_GROUNDED_DATASET', None)
    if RELAX_ROSTER:
        e['ATC_RELAX_ROSTER'] = '1'
    else:
        e.pop('ATC_RELAX_ROSTER', None)
    return e

env = _train_env()

if RUN_SFT_PHASE:
    sft_cmd = [
        sys.executable, 'training/train_sft.py',
        '--model', MODEL_NAME,
        '--output_dir', str(SFT_OUTPUT_DIR),
        '--n_episodes', str(SFT_N_EPISODES),
        '--max_steps', str(SFT_MAX_STEPS),
        '--batch_size', str(SFT_BATCH),
        '--grad_accum', str(SFT_GRAD_ACCUM),
        '--seed', str(SEED),
        '--eval_split', str(SFT_EVAL_SPLIT),
        '--early_stop_patience', str(SFT_EARLY_STOP_PATIENCE),
    ]
    if CURRICULUM_STATE:
        sft_cmd.extend(['--curriculum_state', str(CURRICULUM_STATE)])
    print('SFT:', ' '.join(sft_cmd))
    subprocess.run(sft_cmd, check=True, env=env)

_sft = SFT_OUTPUT_DIR if (SFT_OUTPUT_DIR / 'adapter_config.json').is_file() else None
if RUN_SFT_PHASE and _sft is None:
    raise RuntimeError('SFT expected but adapter_config.json missing')

train_cmd = [
    sys.executable, 'training/train_grpo.py',
    '--model',         MODEL_NAME,
    '--output_dir',    str(OUTPUT_DIR),
    '--episodes',      str(EPISODES),
    '--n_generations', str(N_GENERATIONS),
    '--batch_size',    str(BATCH_SIZE),
    '--grad_accum',    str(GRAD_ACCUM),
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--temperature',   str(TEMPERATURE),
    '--logging_steps', str(LOGGING_STEPS),
    '--eval_episodes', str(EVAL_EPISODES),
    '--lora_rank',     str(LORA_RANK),
    '--seed',          str(SEED),
    '--training_mode', str(TRAINING_MODE),
]
if USE_GROUNDED_CURRICULUM:
    train_cmd.append('--grounded_curriculum')
    if STATIC_GROUNDED_DATASET:
        train_cmd.append('--static_grounded_dataset')
    if CURRICULUM_STATE:
        train_cmd.extend(['--curriculum_state', str(CURRICULUM_STATE)])
if SKIP_MAIN_EVAL:
    train_cmd.append('--no_eval')
if _sft is not None:
    train_cmd.extend(['--adapter_in', str(_sft)])


# ── Live reward tracking ──────────────────────────────────────────────────────
# TRL 0.16 logs every logging_steps steps in dict format:
#   {'loss': 0.012, 'reward': 0.23, 'reward/AMAN': 0.31, 'step': 10, ...}
reward_history  = defaultdict(list)
loss_history    = []
step_history    = []
last_print_step = -1

_re_reward = re.compile(r"'reward(?:/([\w]+))?':\s*([-\d.eE+]+)")
_re_loss   = re.compile(r"'loss':\s*([-\d.eE+]+)")
_re_step   = re.compile(r"'step':\s*(\d+)")

def _bar(val, lo=-1.0, hi=1.0, width=20):
    """ASCII bar for a reward value in [lo, hi]."""
    frac   = max(0.0, min(1.0, (val - lo) / (hi - lo)))
    filled = int(frac * width)
    return f'[{"#" * filled}{"-" * (width - filled)}] {val:+.3f}'

def _live_summary(step):
    elapsed_s = time.time() - t0
    print(f'\n── step {step:4d}  elapsed {elapsed_s/60:.1f} min ──────────────────')
    if loss_history:
        print(f'  loss       : {loss_history[-1]:.5f}')
    for role in ('AMAN', 'DMAN', 'GENERATOR', 'SUPERVISOR', 'composite'):
        vals = reward_history.get(role, [])
        if not vals:
            continue
        recent = vals[-min(20, len(vals)):]
        mean_r = sum(recent) / len(recent)
        trend  = ''
        if len(vals) >= 40:
            half  = len(vals) // 2
            early = sum(vals[:half]) / half
            late  = sum(vals[half:]) / (len(vals) - half)
            trend = '  ↑' if late > early + 0.03 else ('  ↓' if late < early - 0.03 else '  →')
        print(f'  {role:10s} : {_bar(mean_r)}{trend}')
    print()

# ── Launch with live stdout streaming ────────────────────────────────────────
# env from _train_env()
env['PYTHONUNBUFFERED'] = '1'   # training script flushes every line immediately

print('Command:', ' '.join(train_cmd))
print('='*60)
t0   = time.time()
proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr so nothing is hidden
    text=True,
    bufsize=1,                  # line-buffered
    env=env,
)

for raw_line in proc.stdout:
    line = raw_line.rstrip('\n')
    print(line, flush=True)                    # echo every line immediately

    # Parse TRL dict-format log lines
    if "'reward'" in line or "'loss'" in line:
        m_step = _re_step.search(line)
        m_loss = _re_loss.search(line)
        if m_loss:
            loss_history.append(float(m_loss.group(1)))
        for m in _re_reward.finditer(line):
            role = m.group(1) or 'composite'
            reward_history[role].append(float(m.group(2)))
        if m_step:
            s = int(m_step.group(1))
            if s != last_print_step:
                last_print_step = s
                step_history.append(s)
                _live_summary(s)

proc.wait()
elapsed = time.time() - t0
print('='*60)

# ── Final summary ─────────────────────────────────────────────────────────────
if reward_history:
    print('\n=== FINAL REWARD SUMMARY ===')
    for role in ('AMAN', 'DMAN', 'GENERATOR', 'SUPERVISOR', 'composite'):
        vals = reward_history.get(role, [])
        if not vals:
            continue
        n   = len(vals)
        avg = sum(vals) / n
        fq  = sum(vals[:max(1, n//4)]) / max(1, n//4)
        lq  = sum(vals[max(0, 3*n//4):]) / max(1, n - 3*n//4)
        trend = '↑' if lq > fq + 0.03 else ('↓' if lq < fq - 0.03 else '→')
        print(f'  {role:12s}: mean={avg:+.3f}  first_q={fq:+.3f}  last_q={lq:+.3f}  {trend}')

status = 'complete' if proc.returncode == 0 else f'FAILED (exit code {proc.returncode})'
print(f'\nTraining {status} in {elapsed/60:.1f} min')


## 9. Plot reward curves

All plots saved to `OUTPUT_DIR/plots/*.png` — embed in README.

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from pathlib import Path

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
curves_path = OUTPUT_DIR / 'reward_curves.json'

if not curves_path.exists():
    print(f'reward_curves.json not found at {curves_path}')
else:
    with open(curves_path) as f:
        curves = json.load(f)

    def _ema(vals, span=15):
        if len(vals) < 2: return vals
        alpha, ema, out = 2/(span+1), vals[0], []
        for v in vals:
            ema = alpha*v + (1-alpha)*ema
            out.append(ema)
        return out

    # ── Plot 1: Per-role reward curves ────────────────────────────────────────
    role_colors = {'AMAN':'#2196F3','DMAN':'#4CAF50','GENERATOR':'#FF9800','SUPERVISOR':'#9C27B0'}
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, (role, color) in zip(axes.flatten(), role_colors.items()):
        raw = curves.get(role, [])
        if not raw:
            ax.text(0.5, 0.5, f'No {role} data', ha='center', va='center',
                    transform=ax.transAxes, color='grey')
            ax.set_title(role); continue
        steps = range(len(raw))
        ax.plot(steps, raw,       alpha=0.25, color=color, lw=0.8, label='raw')
        ax.plot(steps, _ema(raw), color=color, lw=2.0, label='EMA-15')
        ax.axhline(0, color='black', lw=0.5, ls='--')
        ax.set_xlabel('Training step'); ax.set_ylabel('Reward')
        ax.set_title(f'{role} reward over training', fontweight='bold')
        ax.set_ylim(-1.05, 1.05); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    fig.suptitle('ATC Multi-Agent GRPO — Per-Role Reward Curves', fontsize=14, fontweight='bold', y=1.01)
    fig.tight_layout()
    p1 = PLOTS_DIR / 'per_role_rewards.png'
    fig.savefig(p1, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'Saved: {p1}'); display(Image(filename=str(p1)))

    # ── Plot 2: Composite reward + rolling std (GRPO health check) ────────────
    composite = curves.get('composite', [])
    if composite:
        w = 20
        stds = [float(np.std(composite[max(0,i-w):i+1])) for i in range(len(composite))]
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        steps = range(len(composite))
        ax1.plot(steps, composite,       alpha=0.25, color='#607D8B', lw=0.8)
        ax1.plot(steps, _ema(composite), color='#607D8B', lw=2.0, label='EMA-15')
        ax1.axhline(0, color='black', lw=0.6, ls='--')
        ax1.set_ylabel('Composite reward'); ax1.set_ylim(-1.05, 1.05)
        ax1.set_title('Composite reward (all roles)', fontweight='bold')
        ax1.grid(alpha=0.3); ax1.legend(fontsize=9)
        ax2.plot(steps, stds, color='#E91E63', lw=1.5, label=f'rolling std (w={w})')
        ax2.axhline(0.01, color='red', lw=0.8, ls=':', label='collapse threshold 0.01')
        ax2.set_xlabel('Training step')
        ax2.set_ylabel('Within-group std')
        ax2.set_title('Reward diversity — must stay > 0.01 for GRPO gradient', fontweight='bold')
        ax2.grid(alpha=0.3); ax2.legend(fontsize=9)
        fig.tight_layout()
        p2 = PLOTS_DIR / 'composite_and_std.png'
        fig.savefig(p2, dpi=150, bbox_inches='tight'); plt.close(fig)
        print(f'Saved: {p2}'); display(Image(filename=str(p2)))

    # ── Plot 3: First-quarter vs last-quarter per role ────────────────────────
    roles_ok = [r for r in ['AMAN','DMAN','GENERATOR','SUPERVISOR'] if len(curves.get(r,[])) >= 8]
    if roles_ok:
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(roles_ok))
        fq, lq = [], []
        for role in roles_ok:
            r = curves[role]; n = len(r)
            fq.append(float(np.mean(r[:max(1,n//4)]))); lq.append(float(np.mean(r[max(0,3*n//4):])))
        b1 = ax.bar(x-0.2, fq, 0.35, label='First quarter', color='#90CAF9', edgecolor='#1565C0')
        b2 = ax.bar(x+0.2, lq, 0.35, label='Last quarter',  color='#A5D6A7', edgecolor='#2E7D32')
        for b in list(b1)+list(b2):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                    f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(roles_ok, fontsize=11)
        ax.set_ylabel('Mean reward')
        ax.set_title('First vs last quarter reward per role — evidence of learning', fontweight='bold')
        ax.axhline(0, color='black', lw=0.6, ls='--'); ax.legend(); ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        p3 = PLOTS_DIR / 'first_vs_last_quarter.png'
        fig.savefig(p3, dpi=150, bbox_inches='tight'); plt.close(fig)
        print(f'Saved: {p3}'); display(Image(filename=str(p3)))

## 10. Before vs after comparison

Heuristic baseline (no LLM) · base model (untrained LoRA) · trained model

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

def _load(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else None

baseline = _load(OUTPUT_DIR / 'baseline_metrics.json')
base_mdl = _load(OUTPUT_DIR / 'base_model_metrics.json')
trained  = _load(OUTPUT_DIR / 'trained_model_metrics.json')

if not any([baseline, base_mdl, trained]):
    print('No eval metrics found. Run training without --no_eval.')
else:
    keys = [
        ('mean_composite',   'Composite score'),
        ('mean_aman_reward', 'AMAN reward'),
        ('mean_dman_reward', 'DMAN reward'),
        ('mean_conflicts',   'Avg conflicts (lower=better)'),
    ]
    print(f'{"Metric":25s} {"Heuristic":>12} {"Base model":>12} {"Trained":>12}')
    print('-'*64)
    for k, label in keys:
        bv = f'{baseline.get(k,float("nan")):.3f}' if baseline else 'N/A'
        mv = f'{base_mdl.get(k, float("nan")):.3f}' if base_mdl else 'N/A'
        tv = f'{trained.get(k,  float("nan")):.3f}' if trained  else 'N/A'
        print(f'{label:25s} {bv:>12} {mv:>12} {tv:>12}')

    sources = []
    if baseline: sources.append(('Heuristic\nbaseline',     baseline, '#B0BEC5'))
    if base_mdl: sources.append(('Base model\n(no finetune)', base_mdl, '#90CAF9'))
    if trained:  sources.append(('Trained\nmodel',           trained,  '#A5D6A7'))

    if sources:
        plot_keys = [(k, lbl) for k, lbl in keys if 'conflict' not in k]
        fig, axes = plt.subplots(1, len(plot_keys), figsize=(4*len(plot_keys), 5))
        if len(plot_keys) == 1: axes = [axes]
        for ax, (k, label) in zip(axes, plot_keys):
            vals   = [s[1].get(k, 0.0) for s in sources]
            labels = [s[0] for s in sources]
            colors = [s[2] for s in sources]
            bars = ax.bar(labels, vals, color=colors, edgecolor='#333', width=0.55)
            for b, v in zip(bars, vals):
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=10)
            ax.set_title(label, fontweight='bold'); ax.set_ylabel('Score')
            ax.set_ylim(0, 1.15); ax.grid(axis='y', alpha=0.3)
        fig.suptitle('Before vs After GRPO Training', fontsize=14, fontweight='bold', y=1.03)
        fig.tight_layout()
        p_eval = PLOTS_DIR / 'eval_comparison.png'
        fig.savefig(p_eval, dpi=150, bbox_inches='tight'); plt.close(fig)
        print(f'\nSaved: {p_eval}'); display(Image(filename=str(p_eval)))

## 11. Summary

In [ ]:
print('='*60)
print('  ATC GRPO Training — Run Complete')
print('='*60)
print(f'  Model:         {MODEL_NAME}')
print(f'  Episodes:      {EPISODES}')
print(f'  Group size:    {N_GENERATIONS}')
print(f'  LoRA rank:     {LORA_RANK}')
print(f'  Output:        {OUTPUT_DIR}')
print(f'  Plots:         {PLOTS_DIR}')
print()
print('  Output files:')
all_files = sorted(OUTPUT_DIR.rglob('*.json')) + sorted(PLOTS_DIR.rglob('*.png'))
for f in all_files:
    print(f'    {f.relative_to(OUTPUT_DIR)}')
print()
print('  Reward fixes applied:')
print('    [1] delay_eff=0 for fully-unscheduled plans (was 1.0)')
print('    [2] Parse failures get content-aware [-0.80, -0.62] (was fixed -0.8)')
print('    [3] TEMPERATURE=0.9 for diverse GRPO rollouts (was 0.7)')
print('='*60)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `No module named 'mergekit'` | Re-run install cell (Step 2b installs mergekit for TRL GRPO) |
| `retry` ImportError | Re-run Cell 3 — do NOT add `--no-deps` to the unsloth install line |
| CUDA OOM | Set `N_GENERATIONS=2`; use `Qwen2.5-1.5B-Instruct` |
| `GRPOConfig` keyword error | Confirm `trl==0.16.0` printed in Cell 3 output |
| Rewards still constant | `os.environ['ATC_REWARD_TRACE']='1'` before training to see component values |
| Disconnect mid-training | Enable Drive mount (`USE_DRIVE=True`) |
| Kaggle: slow training | 2×T4 — keep `BATCH_SIZE=2`, `N_GENERATIONS=4` as-is |

**Reward trace**: `os.environ['ATC_REWARD_TRACE'] = '1'` — prints per-component breakdown per completion.